# Language-model prompting

Builds the structured prompts, parses and validates a model reply, and shows how a reply is turned into aspect spans with offsets. **No benchmark is run in this notebook.** Scoring a hosted or local model against the SemEval test set needs a model endpoint and a key, which are not part of the repository; the report states that this approach is not evaluated.

## Setup

Run this on a Colab GPU runtime (*Runtime → Change runtime type → GPU*). The cell clones the repository, installs the dependencies and checks the environment. Colab keeps its own `torch`, so it is left as it is.

In [ ]:
import os

if not os.path.isdir("aspect-based-sentiment-analysis"):
    !git clone -q https://github.com/Shivam-Shukla0/aspect-based-sentiment-analysis.git
%cd aspect-based-sentiment-analysis
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## Prompts

The prompt text is stored in versioned files; the zero-shot and few-shot formats differ only in the examples.

In [ ]:
from absa.extraction import PROMPT_VERSION, format_few_shot_prompt, format_zero_shot_prompt

print("prompt version:", PROMPT_VERSION)
print(format_zero_shot_prompt("The battery is weak but the screen is excellent."))

## Parsing and validation

A reply is accepted only if it is a JSON array of objects with `aspect` and `sentiment`; each aspect must occur in the review, and repeated mentions map to successive occurrences. A scripted client stands in for a real model here.

In [ ]:
from absa.models import (
    MockLLMClient,
    ValidationReport,
    parse_llm_json,
    retry_llm_inference,
    validate_aspect_predictions,
)

review = "The battery is weak but the screen is excellent. The screen is also bright."
reply = (
    '[{"aspect": "battery", "sentiment": "negative"}, '
    '{"aspect": "screen", "sentiment": "positive"}, '
    '{"aspect": "screen", "sentiment": "positive"}, '
    '{"aspect": "keyboard", "sentiment": "neutral"}]'
)
client = MockLLMClient(default_response=reply)
report = ValidationReport()
raw = retry_llm_inference(lambda: parse_llm_json(client.generate(format_few_shot_prompt(review))), report=report)
for span in validate_aspect_predictions(review, raw, report=report):
    print(span)
print("rejected:", report.invalid_count, report.errors)

## Running a real model

`HttpLLMClient` reads `LLM_API_KEY`, `LLM_MODEL` and `LLM_API_BASE` from the environment; in Colab store them as secrets, never in a cell. Results from such a run must be saved with `python -m scripts.verify_environment --record` before they are quoted.